# Post-OFAT DL provider ensembles (Colab)

DL-аналог OFAT ML ensembles. Ноутбук работает поверх результатов `Post_OFAT_DL_all_models_with_tuning_*_Colab.ipynb`:

- читает `best_scheme_dl_tuning_results.csv` для `gpt`, `ollama`, `qwen`;
- восстанавливает tuned DL-модели на их лучших OFAT-конфигурациях;
- кэширует предсказания на blend-validation, typical holdout и full holdout;
- ищет single / mean / median / inverse-MAE / NNLS / simplex / pair-grid ансамбли отдельно для каждого provider;
- сохраняет leaderboard и best ensemble artifacts на Google Drive.


In [ ]:
# Colab/runtime dependencies for OFAT DL ensembling.
import sys
!{sys.executable} -m pip install -q pandas numpy scipy scikit-learn optuna matplotlib seaborn xgboost


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import ast
import json
import math
import time
import gc
import random
import shutil
from datetime import datetime
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.optimize import nnls, minimize
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV, BayesianRidge, HuberRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBRegressor
    HAS_XGB_FOR_STACKING = True
except Exception:
    HAS_XGB_FOR_STACKING = False

SEED = 2
seed = SEED
TARGET_COL = "duration_hours"
DURATION_CAP = 960
BLEND_TRAIN_FRAC = 0.85
BLEND_FIT_FRAC = 0.5
MAX_SUBSET_SIZE = 4
RUN_STACKERS = True
RUN_ALL_PAIRS = True
RUN_ALL_TRIPLES = True
TOP_REFIT_SPECS = 120

SELECTED_DL_MODELS = [
    "ExcelFormer",
    "FT-Transformer",
    "BiGRU",
    "GrowNet",
    "Trompt",
    "TabTransformer",
    "AutoInt",
    "CNN1D",
    "ResMLP",
    "SAINT",
]

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    drive = None

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    BASE_RUNTIME_DIR = Path("/content")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    BASE_RUNTIME_DIR = Path(".")
    DRIVE_ROOT = Path(".")

RUN_NAME = "post_ofat_dl_provider_ensembles"
ART_DIR = DRIVE_ROOT / "artifacts_post_ofat_dl_provider_ensembles"
ART_DIR.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = ART_DIR / f"run_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
PRED_CACHE_DIR = RUN_DIR / "pred_cache"
PRED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    DRIVE_ROOT / "inputs" / "data_finall_without_TTM.csv",
    DRIVE_ROOT / "data_finall_without_TTM.csv",
    BASE_RUNTIME_DIR / "data_finall_without_TTM.csv",
    Path("./data_finall_without_TTM.csv"),
    Path("/mnt/data/data_finall_without_TTM.csv"),
    Path("./data_finall.csv"),
    Path("/mnt/data/data_finall.csv"),
]

SUMMARY_CANDIDATES = {
    "gpt": [
        DRIVE_ROOT / "artifacts_post_ofat_dl_all_models_with_tuning_gpt" / "best_scheme_dl_tuning_results.csv",
        Path("./artifacts_post_ofat_dl_all_models_with_tuning_gpt") / "best_scheme_dl_tuning_results.csv",
    ],
    "ollama": [
        DRIVE_ROOT / "llm_local_colab_runs" / "ollama_meta_prompt_ofat_fixed_features" / "artifacts_post_ofat_dl_all_models_with_tuning_llama3.2_3b" / "best_scheme_dl_tuning_results.csv",
        Path("./ollama_meta_prompt_ofat_fixed_features") / "artifacts_post_ofat_dl_all_models_with_tuning_llama3.2_3b" / "best_scheme_dl_tuning_results.csv",
    ],
    "qwen": [
        DRIVE_ROOT / "llm_local_colab_runs" / "qwen_meta_prompt_ofat_fixed_features" / "artifacts_post_ofat_dl_all_models_with_tuning_Qwen_Qwen2.5-3B-Instruct" / "best_scheme_dl_tuning_results.csv",
        Path("./qwen_meta_prompt_ofat_fixed_features") / "artifacts_post_ofat_dl_all_models_with_tuning_Qwen_Qwen2.5-3B-Instruct" / "best_scheme_dl_tuning_results.csv",
    ],
}

ARTIFACT_ROOT_CANDIDATES = {
    "gpt": [
        DRIVE_ROOT / "artifacts_llm_meta_prompt_ofat_gpt_v8_batch25",
        DRIVE_ROOT / "artifacts",
        Path("./artifacts_llm_meta_prompt_ofat_gpt_v8_batch25"),
        Path("./artifacts"),
    ],
    "ollama": [
        DRIVE_ROOT / "llm_local_colab_runs" / "ollama_meta_prompt_ofat_fixed_features" / "artifacts_fixed_features_v7_llama3.2_3b",
        Path("./ollama_meta_prompt_ofat_fixed_features") / "artifacts_fixed_features_v7_llama3.2_3b",
        Path("./ollama_local/artifacts_ollama_local_pristine"),
    ],
    "qwen": [
        DRIVE_ROOT / "llm_local_colab_runs" / "qwen_meta_prompt_ofat_fixed_features" / "artifacts_fixed_features_v7_Qwen_Qwen2.5-3B-Instruct",
        Path("./qwen_meta_prompt_ofat_fixed_features") / "artifacts_fixed_features_v7_Qwen_Qwen2.5-3B-Instruct",
        Path("./qwen_local/artifacts_fixed_features_v7_Qwen_Qwen2.5-3B-Instruct"),
    ],
}

print("IN_COLAB:", IN_COLAB)
print("ART_DIR:", ART_DIR)
print("RUN_DIR:", RUN_DIR)
print("Selected DL models:", SELECTED_DL_MODELS)


In [ ]:
def seed_everything(s=SEED):
    random.seed(s)
    np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(s)
    except Exception:
        pass


def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None


def sanitize_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(name))


def metrics(y_true, pred):
    pred = np.clip(np.asarray(pred, dtype=float), 0, None)
    y_true = np.asarray(y_true, dtype=float)
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MdAE": float(median_absolute_error(y_true, pred)),
    }


def parse_best_params(raw):
    if isinstance(raw, dict):
        bp = raw
    elif pd.isna(raw):
        bp = {}
    else:
        text = str(raw).replace("NaN", "null")
        try:
            bp = json.loads(text)
        except Exception:
            bp = ast.literal_eval(str(raw))
    return serialize_params(bp)


def save_df(df: pd.DataFrame, name: str):
    run_path = RUN_DIR / name
    latest_path = ART_DIR / name.replace(".csv", "_latest.csv")
    df.to_csv(run_path, index=False)
    df.to_csv(latest_path, index=False)
    return run_path


def load_data():
    path = first_existing(DATA_CANDIDATES)
    if path is None:
        raise FileNotFoundError(f"Data file not found. Tried: {[str(p) for p in DATA_CANDIDATES]}")
    df = pd.read_csv(path).copy()
    df["source_row_id"] = np.arange(len(df), dtype=int)
    return df, path


def split_data(df):
    split = int(len(df) * 0.8)
    train_full = df.iloc[:split].copy()
    test_full = df.iloc[split:].copy().reset_index(drop=True)
    train_filt = train_full[train_full[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)
    test_typical = test_full[test_full[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)

    meta_train = train_filt.drop(columns=[TARGET_COL], errors="ignore").copy()
    meta_test_full = test_full.drop(columns=[TARGET_COL], errors="ignore").copy()
    y_train = train_filt[TARGET_COL].reset_index(drop=True)
    y_test_full = test_full[TARGET_COL].reset_index(drop=True)
    y_test_typical = test_typical[TARGET_COL].reset_index(drop=True)
    return train_filt, test_full, test_typical, meta_train, meta_test_full, y_train, y_test_full, y_test_typical


FEATURE_COLS = [
    "execution_complexity_1_5",
    "coordination_risk_1_5",
    "testing_effort_1_5",
    "uncertainty_1_5",
    "urgent_delivery_0_1",
    "likely_long_task_0_1",
]


def prepare_model_matrix(meta_df, agg_df):
    merged = meta_df.merge(agg_df, on="source_row_id", how="left").copy()
    for col in FEATURE_COLS:
        if col not in merged.columns:
            merged[col] = 0.0
    drop_cols = [
        c for c in merged.columns
        if c.endswith("__std")
        or c.endswith("__agree")
        or c.endswith("__p1")
        or c.endswith("__var")
        or c in {"raw_text", "latency_s", "mean_latency_s"}
    ]
    merged = merged.drop(columns=drop_cols, errors="ignore")
    merged = merged.drop(columns=[TARGET_COL], errors="ignore")
    merged = merged.select_dtypes(include=[np.number, "bool"]).copy()
    return merged.replace([np.inf, -np.inf], np.nan).fillna(0.0)


def local_config_dir(row):
    provider = row["provider"]
    checked = []
    for root in ARTIFACT_ROOT_CANDIDATES[provider]:
        cfg_dir = Path(root) / row["stage"] / row["config_slug"]
        required = [
            cfg_dir / "aggregated_features__train_core.csv",
            cfg_dir / "aggregated_features__val_fresh.csv",
            cfg_dir / "aggregated_features__test_full.csv",
        ]
        if all(path.exists() for path in required):
            return cfg_dir
        checked.append(cfg_dir)
    raise FileNotFoundError(f"No complete OFAT dir for {provider} {row['stage']} {row['config_slug']}. Checked: {[str(x) for x in checked]}")


def load_feature_space(row, meta_train, meta_test_full, test_full, test_typical):
    cfg_dir = local_config_dir(row)
    agg_train = pd.concat([
        pd.read_csv(cfg_dir / "aggregated_features__train_core.csv"),
        pd.read_csv(cfg_dir / "aggregated_features__val_fresh.csv"),
    ], ignore_index=True)
    agg_test_full = pd.read_csv(cfg_dir / "aggregated_features__test_full.csv")
    x_train = prepare_model_matrix(meta_train, agg_train)
    x_test_full = prepare_model_matrix(meta_test_full, agg_test_full)
    typical_ids = set(test_typical["source_row_id"].tolist())
    x_test_typical = x_test_full.loc[test_full["source_row_id"].isin(typical_ids)].reset_index(drop=True)
    return x_train, x_test_typical, x_test_full, str(cfg_dir)


## DL architecture stack from DL_ensembles / DL_tuning

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import time, math, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


SEED = 2
seed = SEED
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print('Device:', device)

def seed_everything(s=SEED):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

INPUT_DIM = 1

# ═══════════════════════════════════════════════════════════════
#  2. АРХИТЕКТУРЫ  (24 шт.)
# ═══════════════════════════════════════════════════════════════

# ── 2.1  MLP ─────────────────────────────────────────────────

class MLP(nn.Module):
    def __init__(self, d_in, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


# ── 2.2  ResMLP ──────────────────────────────────────────────

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.block(x))

class ResMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU())
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(self.proj(x)))


# ── 2.3  SNN ─────────────────────────────────────────────────

class SNN(nn.Module):
    def __init__(self, d_in, hidden_dims=[256, 128], alpha_dropout=0.1):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.SELU(), nn.AlphaDropout(alpha_dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)


# ── 2.4  GatedMLP ───────────────────────────────────────────

class GatedBlock(nn.Module):
    def __init__(self, d_in, d_out, dropout):
        super().__init__()
        self.linear = nn.Linear(d_in, d_out * 2)
        self.bn = nn.BatchNorm1d(d_out)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.linear(x)
        h1, h2 = h.chunk(2, dim=-1)
        return self.drop(self.bn(h1 * torch.sigmoid(h2)))

class GatedMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        blocks = [GatedBlock(d_in, hidden, dropout)]
        for _ in range(n_blocks - 1):
            blocks.append(GatedBlock(hidden, hidden, dropout))
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(x))


# ── 2.5  GANDALF ─────────────────────────────────────────────

class GANDALF(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.gate_fc = nn.Linear(d_in, d_in)
        layers = [nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        for _ in range(n_blocks - 1):
            layers += [nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        self.dnn = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        gate = torch.sigmoid(self.gate_fc(x))
        return self.head(self.dnn(x * gate))


# ── 2.6  DAE-MLP ────────────────────────────────────────────

class DAEMLP(nn.Module):
    def __init__(self, d_in, hidden=256, latent=64, noise=0.3, dropout=0.3):
        super().__init__()
        self.noise = noise
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, latent), nn.BatchNorm1d(latent), nn.SiLU(),
        )
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.SiLU(), nn.Linear(hidden, d_in))
        self.reg_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(latent, 32), nn.SiLU(), nn.Linear(32, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x_in = x * (torch.rand_like(x) > self.noise).float() if self.training and self.noise > 0 else x
        z = self.encoder(x_in)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(z), x)
        return self.reg_head(z)


# ── 2.7  CNN1D ───────────────────────────────────────────────

class CNN1D(nn.Module):
    def __init__(self, d_in, channels=[64, 128, 64], ks=5, dropout=0.3):
        super().__init__()
        layers = []
        in_ch = 1
        for ch in channels:
            layers += [nn.Conv1d(in_ch, ch, ks, padding=ks // 2),
                       nn.BatchNorm1d(ch), nn.SiLU(), nn.Dropout(dropout)]
            in_ch = ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Linear(channels[-1], 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.conv(x))
        return self.head(x.squeeze(-1))


# ── 2.8  BiGRU ───────────────────────────────────────────────

class BiGRU(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden, num_layers=n_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        _, h = self.gru(x.unsqueeze(-1))
        return self.head(torch.cat([h[-2], h[-1]], dim=1))


# ── 2.9  FT-Transformer ─────────────────────────────────────

class FTTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.10  TabTransformer ────────────────────────────────────

class TabTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.col_emb = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_in * d_model + d_in, mlp_hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1)) + self.col_emb
        out = self.transformer(tok)
        pooled = out.reshape(out.size(0), -1)
        return self.head(torch.cat([pooled, x], dim=1))


# ── 2.11  SAINT ──────────────────────────────────────────────

class SAINTBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.sa_norm = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ia_norm = nn.LayerNorm(d_model)
        self.inter_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model * 4),
                                 nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
    def forward(self, x):
        h = self.sa_norm(x)
        h, _ = self.self_attn(h, h, h)
        x = x + h
        B, F_, D = x.shape
        if B > 1 and self.training:
            xt = x.permute(1, 0, 2)
            h2 = self.ia_norm(xt)
            h2, _ = self.inter_attn(h2, h2, h2)
            x = (xt + h2).permute(1, 0, 2)
        return x + self.ffn(x)

class SAINT(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        self.blocks = nn.ModuleList([SAINTBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        for blk in self.blocks:
            tok = blk(tok)
        return self.head(tok[:, 0])


# ── 2.12  AutoInt ────────────────────────────────────────────

class AutoIntLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.W_res = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        h, _ = self.attn(x, x, x)
        return F.relu(self.norm(h + self.W_res(x)))

class AutoInt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([AutoIntLayer(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in * d_model, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        for layer in self.layers:
            tok = layer(tok)
        return self.head(tok.reshape(tok.size(0), -1))


# ── 2.13  Trompt ─────────────────────────────────────────────

class TromptLayer(nn.Module):
    def __init__(self, d_in, d_model, n_heads, dropout):
        super().__init__()
        self.prompt = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.gate = nn.Linear(d_model, 1)
    def forward(self, feat_emb, x_raw):
        B = feat_emb.shape[0]
        prompted = feat_emb + self.prompt.expand(B, -1, -1)
        h, _ = self.attn(prompted, prompted, prompted)
        h = self.norm1(prompted + h)
        h = self.norm2(h + self.ffn(h))
        weights = torch.softmax(self.gate(h).squeeze(-1), dim=1)
        return h, weights * x_raw

class Trompt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([TromptLayer(d_in, d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        preds = []
        for layer in self.layers:
            tok, lp = layer(tok, x)
            preds.append(lp)
        return self.head(torch.stack(preds).mean(0))


# ── 2.14  ExcelFormer ───────────────────────────────────────

class ExcelFormer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.importance = nn.Parameter(torch.zeros(d_in))
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        mask = torch.sigmoid(self.importance)
        x_masked = x * mask
        tok = self.feat_emb(x_masked.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.15  ARM-Net ────────────────────────────────────────────

class ARMNet(nn.Module):
    def __init__(self, d_in, n_exp=64, hidden=128, order=2, dropout=0.2):
        super().__init__()
        self.order = order
        self.exp_layers = nn.ModuleList([nn.Linear(d_in, n_exp) for _ in range(order)])
        self.gate = nn.Sequential(nn.Linear(d_in, n_exp * order), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.BatchNorm1d(n_exp * order), nn.Linear(n_exp * order, hidden),
            nn.SiLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x):
        interactions = []
        for i, layer in enumerate(self.exp_layers):
            h = layer(x)
            if i > 0:
                h = h * interactions[-1]
            interactions.append(F.softplus(h))
        cat = torch.cat(interactions, dim=1)
        return self.head(cat * self.gate(x))


# ── 2.16  NODE ───────────────────────────────────────────────

class NODE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_weights = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.01)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        choices = torch.sigmoid(torch.einsum('bj,tdj->btd', x, self.feat_weights) - self.thresholds)
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        return self.drop(tree_out).mean(dim=-1, keepdim=True)


# ── 2.17  GRANDE ─────────────────────────────────────────────

class GRANDE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_logits = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.tree_weights = nn.Parameter(torch.ones(n_trees) / n_trees)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        feat_sel = F.softmax(self.feat_logits, dim=-1)
        proj = torch.einsum('bj,tdj->btd', x, feat_sel)
        choices = torch.sigmoid(10.0 * (proj - self.thresholds))
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        w = F.softmax(self.tree_weights, dim=0)
        return self.drop((tree_out * w).sum(dim=-1, keepdim=True))


# ── 2.18  Net-DNF ────────────────────────────────────────────

class NetDNF(nn.Module):
    def __init__(self, d_in, n_formulas=64, n_conjuncts=6, dropout=0.2):
        super().__init__()
        self.feat_sel = nn.Parameter(torch.randn(n_formulas, n_conjuncts, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_formulas, n_conjuncts))
        self.formula_w = nn.Linear(n_formulas, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        sel = torch.sigmoid(self.feat_sel)
        proj = torch.einsum('bj,fcj->bfc', x, sel)
        conjuncts = torch.sigmoid(proj - self.thresholds)
        formulas = conjuncts.prod(dim=-1)
        return self.formula_w(self.drop(formulas))


# ── 2.19  TabNet (simplified) ───────────────────────────────

class TabNet(nn.Module):
    def __init__(self, d_in, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1):
        super().__init__()
        self.n_steps = n_steps
        self.relax = relax
        self.bn_init = nn.BatchNorm1d(d_in)
        self.shared_fc = nn.Linear(d_in, n_d + n_a)
        self.step_fcs = nn.ModuleList([nn.Linear(d_in, n_d + n_a) for _ in range(n_steps)])
        self.attn_fcs = nn.ModuleList([nn.Sequential(nn.Linear(n_a, d_in), nn.BatchNorm1d(d_in)) for _ in range(n_steps)])
        self.head = nn.Linear(n_d, 1)
        self.n_d = n_d
        self.drop = nn.Dropout(dropout)
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x = self.bn_init(x)
        prior_scales = torch.ones(x.shape[0], x.shape[1], device=x.device)
        agg = torch.zeros(x.shape[0], self.n_d, device=x.device)
        entropy_loss = 0.0
        for i in range(self.n_steps):
            h = self.shared_fc(x * prior_scales) + self.step_fcs[i](x * prior_scales)
            h_d, h_a = h[:, :self.n_d], h[:, self.n_d:]
            h_d = F.silu(h_d)
            agg = agg + self.drop(h_d)
            attn = self.attn_fcs[i](h_a)
            attn = F.softmax(attn * prior_scales, dim=-1)
            prior_scales = prior_scales * (self.relax - attn)
            entropy_loss += (-attn * torch.log(attn + 1e-15)).mean()
        if self.training:
            self.aux_loss = entropy_loss / self.n_steps
        return self.head(agg)


# ── 2.20  TabR ───────────────────────────────────────────────

class TabR(nn.Module):
    def __init__(self, d_in, hidden=256, n_layers=3, k=32, dropout=0.3):
        super().__init__()
        self.k = k
        layers = []
        prev = d_in
        for i in range(n_layers):
            h = hidden if i == 0 else hidden // 2
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        self.encoder = nn.Sequential(*layers)
        self.latent_dim = prev
        self.retrieval_head = nn.Sequential(nn.Linear(prev * 2, 128), nn.SiLU(), nn.Dropout(dropout), nn.Linear(128, 1))
        self.direct_head = nn.Linear(prev, 1)
        self._store_z = None
        self._store_y = None
    def build_store(self, x_train, y_train):
        self.eval()
        with torch.no_grad():
            self._store_z = self.encoder(x_train)
            self._store_y = y_train
    def forward(self, x):
        z = self.encoder(x)
        if self._store_z is not None and not self.training:
            dists = torch.cdist(z, self._store_z)
            _, idx = dists.topk(self.k, largest=False)
            nbr_z = self._store_z[idx]
            sim = -dists.gather(1, idx)
            attn = torch.softmax(sim, dim=1).unsqueeze(-1)
            context = (attn * nbr_z).sum(dim=1)
            return self.retrieval_head(torch.cat([z, context], dim=1))
        return self.direct_head(z)


# ── 2.21  GrowNet stages ────────────────────────────────────

class GrowNetStage(nn.Module):
    def __init__(self, d_in, hidden=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in + 1, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 1))
    def forward(self, x, prev_pred):
        return self.net(torch.cat([x, prev_pred], dim=1))


# ── 2.22  SwitchTab ──────────────────────────────────────────

class SwitchTab(nn.Module):
    def __init__(self, d_in, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_in, d_enc), nn.BatchNorm1d(d_enc), nn.SiLU(), nn.Dropout(dropout))
        self.mutual_proj = nn.Linear(d_enc, d_mutual)
        self.salient_proj = nn.Linear(d_enc, d_salient)
        self.decoder = nn.Sequential(nn.Linear(d_mutual + d_salient, d_in))
        self.head = nn.Sequential(nn.Linear(d_mutual + d_salient, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        h = self.encoder(x)
        mutual = self.mutual_proj(h)
        salient = self.salient_proj(h)
        rep = torch.cat([mutual, salient], dim=1)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(rep), x)
        return self.head(rep)


# ── 2.23  PTaRL ──────────────────────────────────────────────

class PTaRL(nn.Module):
    def __init__(self, d_in, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_latent), nn.BatchNorm1d(d_latent), nn.SiLU())
        self.prototypes = nn.Parameter(torch.randn(n_prototypes, d_latent) * 0.1)
        self.head = nn.Sequential(nn.Linear(d_latent * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        z = self.encoder(x)
        sim = F.cosine_similarity(z.unsqueeze(1), self.prototypes.unsqueeze(0), dim=-1)
        attn = F.softmax(sim * 10, dim=1)
        proto_rep = (attn.unsqueeze(-1) * self.prototypes.unsqueeze(0)).sum(dim=1)
        return self.head(torch.cat([z, proto_rep], dim=1))


# ── 2.24  DCN v2 ────────────────────────────────────────────

class CrossLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.W = nn.Linear(d, d, bias=False)
        self.b = nn.Parameter(torch.zeros(d))
    def forward(self, x0, x):
        return x0 * self.W(x) + self.b + x

class DCNv2(nn.Module):
    def __init__(self, d_in, n_cross=3, hidden=256, dropout=0.3):
        super().__init__()
        self.cross_layers = nn.ModuleList([CrossLayer(d_in) for _ in range(n_cross)])
        self.deep = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.SiLU(), nn.Dropout(dropout))
        self.head = nn.Linear(d_in + hidden // 2, 1)
    def forward(self, x):
        x_cross = x
        for cl in self.cross_layers:
            x_cross = cl(x, x_cross)
        return self.head(torch.cat([x_cross, self.deep(x)], dim=1))


# ═══════════════════════════════════════════════════════════════
#  3. ФУНКЦИИ ОБУЧЕНИЯ
# ═══════════════════════════════════════════════════════════════

def train_model(model, epochs=300, lr=1e-3, wd=1e-4, bs=256,
                patience=30, aux_w=0.0, trial=None):
    """Обучение с early stopping. Возвращает (model, val_mae, epochs_used)."""
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    X_v, y_v = X_va_t.to(device), y_va_t.to(device)

    best_val, best_state, wait = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v_mae = loss_fn(model(X_v), y_v).item()
        sched.step(v_mae)
        if v_mae < best_val:
            best_val = v_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
        if trial is not None:
            trial.report(v_mae, ep)
            if trial.should_prune():
                raise optuna.TrialPruned()
    model.load_state_dict(best_state)
    return model, best_val, ep


def train_grownet(n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                  step_size=0.1, bs=256, patience=30, dropout=0.1, trial=None):
    """Gradient boosting с NN base learners. Возвращает (stages, val_mae, total_epochs)."""
    stages = []
    X_v, y_v = X_va_t.to(device), y_va_t.to(device)
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    # Предсказание ансамбля на train/val
    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    best_val, best_n = float("inf"), 0
    total_ep = 0
    for stage_idx in range(n_stages):
        model = GrowNetStage(INPUT_DIM, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
        best_stage_val, best_stage_state, wait = float("inf"), None, 0
        for ep in range(1, 201):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad(); loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                prev_v = ensemble_pred(X_v, stages)
                v_pred = prev_v + step_size * model(X_v, prev_v)
                v_mae = loss_fn(v_pred, y_v).item()
            sched.step(v_mae)
            if v_mae < best_stage_val:
                best_stage_val = v_mae
                best_stage_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break
            total_ep += 1
        model.load_state_dict(best_stage_state)
        stages.append(model)
        # Проверяем ансамбль
        with torch.no_grad():
            ens_val = loss_fn(ensemble_pred(X_v, stages), y_v).item()
        if ens_val < best_val:
            best_val = ens_val
            best_n = len(stages)
        if trial is not None:
            trial.report(ens_val, stage_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
    stages = stages[:best_n]
    return stages, best_val, total_ep


def eval_on_test(model, name=""):
    """Evaluate model on holdout test."""
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t.to(device)).cpu().numpy().flatten()
    mae = mean_absolute_error(ytest, pred)
    rmse = np.sqrt(mean_squared_error(ytest, pred))
    mdae = median_absolute_error(ytest, pred)
    return mae, rmse, mdae


def eval_grownet_on_test(stages, step_size=0.1):
    """Evaluate GrowNet ensemble on holdout test."""
    X_t = X_te_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    mae = mean_absolute_error(ytest, pred)
    rmse = np.sqrt(mean_squared_error(ytest, pred))
    mdae = median_absolute_error(ytest, pred)
    return mae, rmse, mdae


print("24 архитектуры + train_model + train_grownet определены.")


In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

# ═══════════════════════════════════════════════════════════════
#  Baseline: все 24 архитектуры с дефолтными параметрами
# ═══════════════════════════════════════════════════════════════

DEFAULTS = {
    # MLP-семейство
    "MLP":            lambda: MLP(INPUT_DIM, [256, 128], dropout=0.3),
    "ResMLP":         lambda: ResMLP(INPUT_DIM, hidden=256, n_blocks=3, dropout=0.3),
    "SNN":            lambda: SNN(INPUT_DIM, hidden_dims=[256, 128], alpha_dropout=0.1),
    "GatedMLP":       lambda: GatedMLP(INPUT_DIM, hidden=256, n_blocks=3, dropout=0.3),
    "GANDALF":        lambda: GANDALF(INPUT_DIM, hidden=256, n_blocks=3, dropout=0.3),
    "DAE-MLP":        lambda: DAEMLP(INPUT_DIM, hidden=256, latent=64, noise=0.3, dropout=0.3),
    # CNN / RNN
    "CNN1D":          lambda: CNN1D(INPUT_DIM, channels=[64, 128, 64], ks=5, dropout=0.3),
    "BiGRU":          lambda: BiGRU(INPUT_DIM, hidden=64, n_layers=2, dropout=0.3),
    # Transformer / Attention
    "FT-Transformer": lambda: FTTransformer(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
    "TabTransformer": lambda: TabTransformer(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2),
    "SAINT":          lambda: SAINT(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
    "AutoInt":        lambda: AutoInt(INPUT_DIM, d_model=32, n_heads=4, n_layers=3, dropout=0.2),
    "Trompt":         lambda: Trompt(INPUT_DIM, d_model=32, n_heads=4, n_layers=3, dropout=0.2),
    "ExcelFormer":    lambda: ExcelFormer(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
    "ARM-Net":        lambda: ARMNet(INPUT_DIM, n_exp=64, hidden=128, order=2, dropout=0.2),
    # Tree-inspired
    "NODE":           lambda: NODE(INPUT_DIM, n_trees=32, depth=4, dropout=0.0),
    "GRANDE":         lambda: GRANDE(INPUT_DIM, n_trees=32, depth=4, dropout=0.0),
    "Net-DNF":        lambda: NetDNF(INPUT_DIM, n_formulas=64, n_conjuncts=6, dropout=0.2),
    "TabNet":         lambda: TabNet(INPUT_DIM, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1),
    # Retrieval / Special
    "TabR":           lambda: TabR(INPUT_DIM, hidden=256, n_layers=3, k=32, dropout=0.3),
    "SwitchTab":      lambda: SwitchTab(INPUT_DIM, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3),
    "PTaRL":          lambda: PTaRL(INPUT_DIM, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3),
    "DCNv2":          lambda: DCNv2(INPUT_DIM, n_cross=3, hidden=256, dropout=0.3),
}

AUX_W = {"DAE-MLP": 0.1, "TabNet": 0.01, "SwitchTab": 0.1}
def build_arch_from_params(arch_name: str, bp: dict):
    if arch_name == "MLP":
        dims = [max(int(bp["first_hidden"]) // (2 ** i), 32) for i in range(int(bp["n_layers"]))]
        return MLP(INPUT_DIM, dims, bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ResMLP":
        return ResMLP(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SNN":
        dims = [max(int(bp["first_hidden"]) // (2 ** i), 32) for i in range(int(bp["n_layers"]))]
        return SNN(INPUT_DIM, dims, bp["alpha_dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GatedMLP":
        return GatedMLP(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GANDALF":
        return GANDALF(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "DAE-MLP":
        return DAEMLP(INPUT_DIM, bp["hidden"], bp["latent"], bp["noise"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "CNN1D":
        nc = int(bp["n_conv"]); bc = int(bp["base_ch"])
        chs = [bc * (2 ** i) for i in range(nc // 2 + 1)]
        chs += [bc * (2 ** i) for i in range(nc // 2 - 1, -1, -1)]
        chs = chs[:nc]
        return CNN1D(INPUT_DIM, chs, bp["ks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "BiGRU":
        return BiGRU(INPUT_DIM, bp["hidden"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "FT-Transformer":
        return FTTransformer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabTransformer":
        return TabTransformer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["mlp_hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SAINT":
        return SAINT(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "AutoInt":
        return AutoInt(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "Trompt":
        return Trompt(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ExcelFormer":
        return ExcelFormer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ARM-Net":
        return ARMNet(INPUT_DIM, bp["n_exp"], bp["hidden"], bp["order"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "NODE":
        return NODE(INPUT_DIM, bp["n_trees"], bp["depth"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GRANDE":
        return GRANDE(INPUT_DIM, bp["n_trees"], bp["depth"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "Net-DNF":
        return NetDNF(INPUT_DIM, bp["n_formulas"], bp["n_conjuncts"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabNet":
        return TabNet(INPUT_DIM, bp["n_steps"], bp["n_d"], bp["n_a"], bp["relax"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabR":
        return TabR(INPUT_DIM, bp["hidden"], bp["n_layers"], bp["k"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SwitchTab":
        return SwitchTab(INPUT_DIM, bp["d_enc"], bp["d_mutual"], bp["d_salient"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "PTaRL":
        return PTaRL(INPUT_DIM, bp["n_prototypes"], bp["d_latent"], bp["hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "DCNv2":
        return DCNv2(INPUT_DIM, bp["n_cross"], bp["hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    else:
        raise KeyError(f"Unknown architecture: {arch_name}")

def serialize_params(params: dict):
    clean = {}
    for k, v in params.items():
        if isinstance(v, (np.integer,)):
            clean[k] = int(v)
        elif isinstance(v, (np.floating,)):
            clean[k] = float(v)
        else:
            clean[k] = v
    return clean

In [ ]:
def clear_device_cache():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()

def calc_reg_metrics(y_true, pred):
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MdAE": float(median_absolute_error(y_true, pred)),
    }

def eval_on_typical_holdout(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_on_full_holdout(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_stress_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def eval_grownet_on_typical_holdout(stages, step_size=0.1):
    X_t = X_te_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_grownet_on_full_holdout(stages, step_size=0.1):
    X_t = X_te_stress_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def train_model_fulltrain(model, epochs=100, lr=1e-3, wd=1e-4, bs=256, aux_w=0.0):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_full_t, y_full_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
    return model

def eval_fulltrain_model_typical(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_full_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_fulltrain_model_full(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_fullscale_stress_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def train_grownet_fulltrain(n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                            step_size=0.1, bs=256, epochs_per_stage=50, dropout=0.1):
    stages = []
    ds = TensorDataset(X_full_t, y_full_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    for stage_idx in range(n_stages):
        model = GrowNetStage(INPUT_DIM, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs_per_stage, 1), eta_min=1e-6)

        for ep in range(epochs_per_stage):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad()
                loss.backward()
                opt.step()
            sched.step()

        stages.append(model)
    return stages

def eval_fulltrain_grownet_typical(stages, step_size=0.1):
    X_t = X_te_full_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_fulltrain_grownet_full(stages, step_size=0.1):
    X_t = X_te_fullscale_stress_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

print("Full-holdout helpers defined.")

In [ ]:
def setup_dl_for_ensemble(X_train_all, y_train_all, X_test_typical, y_test_typical, X_test_full, y_test_full):
    global X_dl_tr, y_dl_tr, X_dl_va, y_dl_va, X_dl_te, X_full_s, X_te_full_s
    global X_tr_t, y_tr_t, X_va_t, y_va_t, X_te_t, X_te_stress_t
    global X_full_t, y_full_t, X_te_full_t, X_te_fullscale_stress_t
    global INPUT_DIM, sc, sc_full, Xtrain, ytrain, Xtest, ytest, ytest_typical, ytest_full

    Xtrain = X_train_all.reset_index(drop=True).copy()
    ytrain = pd.Series(y_train_all).reset_index(drop=True)
    Xtest = X_test_typical.reset_index(drop=True).copy()
    ytest = pd.Series(y_test_typical).reset_index(drop=True)
    ytest_typical = ytest.copy()
    ytest_full = pd.Series(y_test_full).reset_index(drop=True)

    blend_start = int(len(Xtrain) * BLEND_TRAIN_FRAC)
    if blend_start <= 0 or blend_start >= len(Xtrain):
        raise ValueError("Bad BLEND_TRAIN_FRAC")

    Xtr_arr = Xtrain.values.astype(np.float32)
    ytr_arr = ytrain.values.astype(np.float32)
    Xte_typ_arr = Xtest.values.astype(np.float32)
    Xte_full_arr = X_test_full.values.astype(np.float32)

    X_dl_tr = Xtr_arr[:blend_start]
    y_dl_tr = ytr_arr[:blend_start]
    X_dl_va = Xtr_arr[blend_start:]
    y_dl_va = ytr_arr[blend_start:]

    sc = StandardScaler()
    X_dl_tr = sc.fit_transform(X_dl_tr)
    X_dl_va = sc.transform(X_dl_va)
    X_dl_te = sc.transform(Xte_typ_arr)
    X_dl_te_full = sc.transform(Xte_full_arr)

    sc_full = StandardScaler()
    X_full_s = sc_full.fit_transform(Xtr_arr)
    X_te_full_s = sc_full.transform(Xte_typ_arr)
    X_te_fullscale_stress = sc_full.transform(Xte_full_arr)

    for arr in [X_dl_tr, X_dl_va, X_dl_te, X_dl_te_full, X_full_s, X_te_full_s, X_te_fullscale_stress]:
        np.nan_to_num(arr, copy=False)

    X_tr_t = torch.from_numpy(X_dl_tr)
    y_tr_t = torch.from_numpy(y_dl_tr).unsqueeze(1)
    X_va_t = torch.from_numpy(X_dl_va)
    y_va_t = torch.from_numpy(y_dl_va).unsqueeze(1)
    X_te_t = torch.from_numpy(X_dl_te)
    X_te_stress_t = torch.from_numpy(X_dl_te_full)
    X_full_t = torch.from_numpy(X_full_s)
    y_full_t = torch.from_numpy(ytr_arr).unsqueeze(1)
    X_te_full_t = torch.from_numpy(X_te_full_s)
    X_te_fullscale_stress_t = torch.from_numpy(X_te_fullscale_stress)
    INPUT_DIM = X_dl_tr.shape[1]


def _torch_pred(model, X_tensor):
    model.eval()
    with torch.no_grad():
        pred = model(X_tensor.to(device)).cpu().numpy().reshape(-1)
    return np.clip(pred, 0, None)


def _grownet_pred(stages, X_tensor, step_size=0.1):
    X_t = X_tensor.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    return np.clip(p.cpu().numpy().reshape(-1), 0, None)


def rebuild_one_tuned_row(row, meta_train, meta_test_full, y_train, y_test_full, y_test_typical, test_full, test_typical):
    x_train, x_test_typical, x_test_full, cfg_dir = load_feature_space(row, meta_train, meta_test_full, test_full, test_typical)
    setup_dl_for_ensemble(x_train, y_train, x_test_typical, y_test_typical, x_test_full, y_test_full)

    arch = row["model_norm"]
    best_params = parse_best_params(row["best_params"])
    epochs_hint = max(50, int(row.get("epochs_used", row.get("epochs", 100)) or 100))
    seed_everything(SEED)
    t0 = time.time()

    if arch == "GrowNet":
        n_stages = int(best_params["n_stages"])
        step_size = float(best_params["step_size"])
        stages, val_mae, split_epochs = train_grownet(
            n_stages=n_stages,
            hidden=int(best_params["hidden"]),
            lr=float(best_params["lr"]),
            wd=float(best_params["wd"]),
            step_size=step_size,
            bs=int(best_params["bs"]),
            patience=30,
            dropout=float(best_params["dropout"]),
        )
        pred_blend = _grownet_pred(stages, X_va_t, step_size)

        epochs_per_stage = max(20, math.ceil(epochs_hint / max(n_stages, 1)))
        full_stages = train_grownet_fulltrain(
            n_stages=n_stages,
            hidden=int(best_params["hidden"]),
            lr=float(best_params["lr"]),
            wd=float(best_params["wd"]),
            step_size=step_size,
            bs=int(best_params["bs"]),
            epochs_per_stage=epochs_per_stage,
            dropout=float(best_params["dropout"]),
        )
        pred_typical = _grownet_pred(full_stages, X_te_full_t, step_size)
        pred_full = _grownet_pred(full_stages, X_te_fullscale_stress_t, step_size)
        epochs_used = int(split_epochs)
        n_params = int(sum(sum(p.numel() for p in s.parameters()) for s in stages))
    else:
        model, aux_w = build_arch_from_params(arch, best_params)
        model, val_mae, epochs_used = train_model(
            model,
            epochs=300,
            lr=float(best_params["lr"]),
            wd=float(best_params["wd"]),
            bs=int(best_params["bs"]),
            patience=30,
            aux_w=aux_w,
        )
        if arch == "TabR":
            model.build_store(X_tr_t.to(device), y_tr_t.to(device))
        pred_blend = _torch_pred(model, X_va_t)

        full_model, aux_w = build_arch_from_params(arch, best_params)
        full_model = train_model_fulltrain(
            full_model,
            epochs=epochs_hint,
            lr=float(best_params["lr"]),
            wd=float(best_params["wd"]),
            bs=int(best_params["bs"]),
            aux_w=aux_w,
        )
        if arch == "TabR":
            full_model.build_store(X_full_t.to(device), y_full_t.to(device))
        pred_typical = _torch_pred(full_model, X_te_full_t)
        pred_full = _torch_pred(full_model, X_te_fullscale_stress_t)
        n_params = int(sum(p.numel() for p in model.parameters()))

    elapsed = time.time() - t0
    clear_device_cache()
    base_id = f"{row['provider']}::{row['stage']}::{row['config_slug']}::{arch}"
    return {
        "base_id": base_id,
        "provider": row["provider"],
        "model": arch,
        "stage": row["stage"],
        "config_slug": row["config_slug"],
        "config_dir": cfg_dir,
        "n_params": n_params,
        "epochs_used": int(epochs_used),
        "rebuild_seconds": round(elapsed, 1),
        "pred_blend_val": pred_blend,
        "pred_typical": pred_typical,
        "pred_full": pred_full,
        "single_blend_MAE": metrics(y_dl_va, pred_blend)["MAE"],
        "single_typical_MAE": metrics(y_test_typical, pred_typical)["MAE"],
        "single_full_MAE": metrics(y_test_full, pred_full)["MAE"],
    }


## Ensemble utilities

In [ ]:
def clip_pred(pred):
    return np.clip(np.asarray(pred, dtype=float), 0, None)


def mae(y, p):
    return float(mean_absolute_error(np.asarray(y, dtype=float), clip_pred(p)))


def weighted_average(preds, weights):
    mat = np.vstack([np.asarray(p, dtype=float) for p in preds])
    w = np.asarray(weights, dtype=float)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.clip(w, 0, None)
    if w.sum() <= 0:
        w = np.ones(len(preds), dtype=float)
    w = w / w.sum()
    return clip_pred(np.average(mat, axis=0, weights=w))


def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.where(np.isfinite(w), w, 0.0)
    w = np.clip(w, 0, None)
    if w.sum() <= 0:
        return np.ones_like(w) / len(w)
    return w / w.sum()


def fit_weights(pred_fit_df, y_fit, scheme):
    X = pred_fit_df.values.astype(float)
    if scheme == "equal":
        return np.ones(X.shape[1]) / X.shape[1]
    errs = np.array([mae(y_fit, pred_fit_df[c].values) for c in pred_fit_df.columns], dtype=float)
    if scheme == "inverse_mae":
        return normalize_weights(1.0 / np.maximum(errs, 1e-9))
    if scheme == "inverse_mae_sq":
        return normalize_weights(1.0 / np.maximum(errs, 1e-9) ** 2)
    if scheme == "nnls":
        w, _ = nnls(X, np.asarray(y_fit, dtype=float))
        return normalize_weights(w)
    if scheme == "simplex_mae":
        n = X.shape[1]
        x0 = np.ones(n) / n
        bounds = [(0.0, 1.0)] * n
        cons = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}
        res = minimize(lambda w: mae(y_fit, X @ w), x0=x0, method="SLSQP", bounds=bounds, constraints=[cons], options={"maxiter": 500, "ftol": 1e-8})
        return normalize_weights(res.x if res.success else x0)
    raise KeyError(scheme)


def build_stacker(name):
    if name == "linear":
        return LinearRegression()
    if name == "positive_linear":
        return LinearRegression(positive=True)
    if name == "ridge":
        return RidgeCV(alphas=np.logspace(-4, 4, 25))
    if name == "bayes":
        return BayesianRidge()
    if name == "huber":
        return HuberRegressor(max_iter=1000, epsilon=1.35)
    if name == "gbr":
        return GradientBoostingRegressor(loss="absolute_error", random_state=SEED, n_estimators=250, learning_rate=0.05, max_depth=2, min_samples_leaf=5)
    if name == "xgb":
        if not HAS_XGB_FOR_STACKING:
            raise RuntimeError("xgboost unavailable")
        return XGBRegressor(objective="reg:absoluteerror", eval_metric="mae", n_estimators=350, max_depth=2, learning_rate=0.03, subsample=0.9, colsample_bytree=0.9, random_state=SEED, tree_method="hist", verbosity=0)
    raise KeyError(name)


def prediction_frames(payloads):
    model_ids = [p["base_id"] for p in payloads]
    blend_df = pd.DataFrame({p["base_id"]: p["pred_blend_val"] for p in payloads})
    typical_df = pd.DataFrame({p["base_id"]: p["pred_typical"] for p in payloads})
    full_df = pd.DataFrame({p["base_id"]: p["pred_full"] for p in payloads})
    return model_ids, blend_df, typical_df, full_df


def generate_ensemble_specs(ranked_models):
    specs = []
    for m in ranked_models:
        specs.append({"family": "single", "name": "single", "models": [m]})
    if len(ranked_models) >= 2:
        for k in range(2, len(ranked_models) + 1):
            topk = ranked_models[:k]
            specs.append({"family": "aggregate", "name": f"top{k}_mean", "models": topk, "method": "mean"})
            specs.append({"family": "aggregate", "name": f"top{k}_median", "models": topk, "method": "median"})
            for scheme in ["inverse_mae", "inverse_mae_sq", "nnls", "simplex_mae"]:
                specs.append({"family": "weighted", "name": f"top{k}_{scheme}", "models": topk, "scheme": scheme})
        if RUN_ALL_PAIRS:
            for a, b in combinations(ranked_models, 2):
                specs.append({"family": "aggregate", "name": "pair_mean", "models": [a, b], "method": "mean"})
                specs.append({"family": "pair_grid", "name": "pair_grid", "models": [a, b]})
        if RUN_ALL_TRIPLES:
            for trio in combinations(ranked_models, 3):
                specs.append({"family": "aggregate", "name": "triple_mean", "models": list(trio), "method": "mean"})
        for r in range(2, min(MAX_SUBSET_SIZE, len(ranked_models)) + 1):
            for combo in combinations(ranked_models, r):
                specs.append({"family": "aggregate", "name": f"subset{r}_mean", "models": list(combo), "method": "mean"})
                specs.append({"family": "weighted", "name": f"subset{r}_inverse_mae", "models": list(combo), "scheme": "inverse_mae"})
        if RUN_STACKERS:
            for k in range(2, len(ranked_models) + 1):
                topk = ranked_models[:k]
                for stacker in ["linear", "positive_linear", "ridge", "bayes", "huber", "gbr", "xgb"]:
                    specs.append({"family": "stacking", "name": f"top{k}_{stacker}", "models": topk, "stacker": stacker})
    return specs


def eval_spec_on_selection(spec, pred_fit_df, y_fit, pred_sel_df, y_sel):
    models = spec["models"]
    fam = spec["family"]
    if fam == "single":
        pred = pred_sel_df[models[0]].values
        return {"spec": spec, "selection_MAE": mae(y_sel, pred), "weights": [1.0]}
    if fam == "aggregate":
        mat = pred_sel_df[models].values.astype(float)
        if spec["method"] == "mean":
            pred = mat.mean(axis=1)
        elif spec["method"] == "median":
            pred = np.median(mat, axis=1)
        else:
            raise KeyError(spec["method"])
        return {"spec": spec, "selection_MAE": mae(y_sel, pred), "weights": []}
    if fam == "weighted":
        w = fit_weights(pred_fit_df[models], y_fit, spec["scheme"])
        pred = pred_sel_df[models].values @ w
        return {"spec": spec, "selection_MAE": mae(y_sel, pred), "weights": w.tolist()}
    if fam == "pair_grid":
        a, b = models
        best = None
        for w0 in np.linspace(0, 1, 101):
            w = np.array([w0, 1 - w0], dtype=float)
            fit_pred = pred_fit_df[[a, b]].values @ w
            score = mae(y_fit, fit_pred)
            if best is None or score < best[0]:
                best = (score, w)
        pred = pred_sel_df[[a, b]].values @ best[1]
        return {"spec": spec, "selection_MAE": mae(y_sel, pred), "weights": best[1].tolist()}
    if fam == "stacking":
        model = build_stacker(spec["stacker"])
        model.fit(pred_fit_df[models].values.astype(float), np.asarray(y_fit, dtype=float))
        pred = model.predict(pred_sel_df[models].values.astype(float))
        return {"spec": spec, "selection_MAE": mae(y_sel, pred), "stacker_model": model, "weights": []}
    raise KeyError(fam)


def predict_spec_final(fitted, pred_fit_full_df, y_fit_full, pred_typical_df, y_typical, pred_full_df, y_full):
    spec = fitted["spec"]
    models = spec["models"]
    fam = spec["family"]
    weights = fitted.get("weights", [])
    if fam == "single":
        pred_t = pred_typical_df[models[0]].values
        pred_f = pred_full_df[models[0]].values
    elif fam == "aggregate":
        mat_t = pred_typical_df[models].values.astype(float)
        mat_f = pred_full_df[models].values.astype(float)
        if spec["method"] == "mean":
            pred_t = mat_t.mean(axis=1); pred_f = mat_f.mean(axis=1)
        elif spec["method"] == "median":
            pred_t = np.median(mat_t, axis=1); pred_f = np.median(mat_f, axis=1)
        else:
            raise KeyError(spec["method"])
    elif fam == "weighted" or fam == "pair_grid":
        w = np.asarray(weights, dtype=float)
        pred_t = pred_typical_df[models].values @ w
        pred_f = pred_full_df[models].values @ w
    elif fam == "stacking":
        # Refit stacker on full blend-val predictions before holdout evaluation.
        model = build_stacker(spec["stacker"])
        model.fit(pred_fit_full_df[models].values.astype(float), np.asarray(y_fit_full, dtype=float))
        pred_t = model.predict(pred_typical_df[models].values.astype(float))
        pred_f = model.predict(pred_full_df[models].values.astype(float))
    else:
        raise KeyError(fam)
    mt = metrics(y_typical, pred_t)
    mf = metrics(y_full, pred_f)
    return {
        "family": fam,
        "name": spec["name"],
        "n_models": len(models),
        "models": models,
        "selection_MAE": float(fitted["selection_MAE"]),
        "MAE_typical": mt["MAE"],
        "RMSE_typical": mt["RMSE"],
        "MdAE_typical": mt["MdAE"],
        "MAE_full": mf["MAE"],
        "RMSE_full": mf["RMSE"],
        "MdAE_full": mf["MdAE"],
        "weights": weights,
    }


## Rebuild tuned OFAT DL models

In [ ]:
df, DATA_PATH = load_data()
train_filt, test_full, test_typical, meta_train, meta_test_full, y_train, y_test_full, y_test_typical = split_data(df)
print("DATA:", DATA_PATH)
print("Train typical:", len(train_filt), "Test typical:", len(test_typical), "Test full:", len(test_full))

provider_frames = []
for provider, paths in SUMMARY_CANDIDATES.items():
    summary_path = first_existing(paths)
    artifact_root = first_existing(ARTIFACT_ROOT_CANDIDATES[provider])
    if summary_path is None or artifact_root is None:
        print(f"[SKIP] {provider}: missing summary or OFAT artifact root")
        continue
    frame = pd.read_csv(summary_path)
    frame["provider"] = provider
    frame["summary_path"] = str(summary_path)
    frame = frame[frame["model_norm"].isin(SELECTED_DL_MODELS)].copy()
    sort_cols = [c for c in ["optuna_best_val_MAE", "refit_val_MAE", "holdout_typical_MAE"] if c in frame.columns]
    frame = frame.sort_values(sort_cols).reset_index(drop=True)
    provider_frames.append(frame)
    print(f"[LOAD] {provider}: rows={len(frame)} summary={summary_path} artifacts={artifact_root}")

if not provider_frames:
    raise FileNotFoundError("No provider DL tuning summaries found. Run/upload Post_OFAT_DL_all_models_with_tuning_* outputs first.")

base_rows = pd.concat(provider_frames, ignore_index=True).reset_index(drop=True)
save_df(base_rows, "ofat_provider_selected_tuned_dl_rows.csv")
display(base_rows[["provider", "model_norm", "stage", "config_slug"] + [c for c in ["optuna_best_val_MAE", "holdout_typical_MAE", "holdout_full_MAE"] if c in base_rows.columns]])

payloads = []
skipped = []
for i, row in base_rows.iterrows():
    print(f"\n[{i+1}/{len(base_rows)}] rebuild {row['provider']} {row['model_norm']} :: {row['stage']} :: {row['config_slug']}")
    try:
        payload = rebuild_one_tuned_row(row, meta_train, meta_test_full, y_train, y_test_full, y_test_typical, test_full, test_typical)
        payloads.append(payload)
        provider_cache = PRED_CACHE_DIR / row["provider"]
        provider_cache.mkdir(parents=True, exist_ok=True)
        base_slug = sanitize_name(payload["base_id"])
        np.save(provider_cache / f"{base_slug}__blend_val.npy", payload["pred_blend_val"])
        np.save(provider_cache / f"{base_slug}__test_typical.npy", payload["pred_typical"])
        np.save(provider_cache / f"{base_slug}__test_full.npy", payload["pred_full"])
        print(f"  blend={payload['single_blend_MAE']:.3f} typ={payload['single_typical_MAE']:.3f} full={payload['single_full_MAE']:.3f} t={payload['rebuild_seconds']:.1f}s")
    except Exception as exc:
        print(f"  SKIP: {type(exc).__name__}: {exc}")
        bad = row.to_dict()
        bad["skip_reason"] = f"{type(exc).__name__}: {exc}"
        skipped.append(bad)
        clear_device_cache()

if skipped:
    save_df(pd.DataFrame(skipped), "ofat_provider_dl_skipped_rows.csv")
if len(payloads) < 2:
    raise RuntimeError("Need at least two rebuilt DL base models for ensembles.")

base_meta = pd.DataFrame([{k: v for k, v in p.items() if not k.startswith("pred_")} for p in payloads])
save_df(base_meta, "ofat_provider_dl_refit_base_metrics.csv")
display(base_meta.sort_values(["provider", "single_blend_MAE"]).head(30))


## Search and evaluate provider ensembles

In [ ]:
leaderboards = []
best_rows = []

for provider in sorted(set(p["provider"] for p in payloads)):
    provider_payloads = [p for p in payloads if p["provider"] == provider]
    if len(provider_payloads) < 2:
        print(f"[SKIP_ENSEMBLE] {provider}: fewer than 2 bases")
        continue
    model_ids, blend_df, typical_df, full_df = prediction_frames(provider_payloads)
    y_blend = np.asarray(y_dl_va, dtype=float)  # same length by construction after last setup; all providers use same split size
    # Safer: derive from current train split length directly.
    blend_start = int(len(y_train) * BLEND_TRAIN_FRAC)
    y_blend = y_train.iloc[blend_start:].to_numpy(dtype=float)

    blend_cut = int(len(y_blend) * BLEND_FIT_FRAC)
    if blend_cut <= 0 or blend_cut >= len(y_blend):
        raise ValueError("Bad BLEND_FIT_FRAC")
    pred_fit_df = blend_df.iloc[:blend_cut].copy()
    pred_sel_df = blend_df.iloc[blend_cut:].copy()
    y_fit = y_blend[:blend_cut]
    y_sel = y_blend[blend_cut:]

    base_rank = pd.DataFrame({
        "base_id": model_ids,
        "fit_MAE": [mae(y_fit, pred_fit_df[m].values) for m in model_ids],
        "selection_MAE": [mae(y_sel, pred_sel_df[m].values) for m in model_ids],
    }).sort_values(["fit_MAE", "selection_MAE"]).reset_index(drop=True)
    ranked_models = base_rank["base_id"].tolist()
    base_rank.to_csv(RUN_DIR / f"{provider}_blend_fit_base_ranking.csv", index=False)
    base_rank.to_csv(ART_DIR / f"{provider}_blend_fit_base_ranking_latest.csv", index=False)

    specs = generate_ensemble_specs(ranked_models)
    search_rows = []
    for idx, spec in enumerate(specs, start=1):
        try:
            fitted = eval_spec_on_selection(spec, pred_fit_df, y_fit, pred_sel_df, y_sel)
            search_rows.append(fitted)
        except Exception as exc:
            if idx <= 10:
                print(f"  spec skip {provider}: {spec.get('family')} {spec.get('name')} {type(exc).__name__}: {exc}")
    search_df = pd.DataFrame([
        {
            "provider": provider,
            "family": r["spec"]["family"],
            "name": r["spec"]["name"],
            "n_models": len(r["spec"]["models"]),
            "models": json.dumps(r["spec"]["models"], ensure_ascii=False),
            "selection_MAE": r["selection_MAE"],
            "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
            "_obj": r,
        }
        for r in search_rows
    ]).sort_values(["selection_MAE", "n_models"]).reset_index(drop=True)
    search_df.drop(columns=["_obj"]).to_csv(RUN_DIR / f"{provider}_ofat_dl_ensemble_search_leaderboard.csv", index=False)
    search_df.drop(columns=["_obj"]).to_csv(ART_DIR / f"{provider}_ofat_dl_ensemble_search_leaderboard_latest.csv", index=False)

    final_rows = []
    for _, sr in search_df.head(TOP_REFIT_SPECS).iterrows():
        out = predict_spec_final(
            sr["_obj"],
            pred_fit_full_df=blend_df,
            y_fit_full=y_blend,
            pred_typical_df=typical_df,
            y_typical=y_test_typical,
            pred_full_df=full_df,
            y_full=y_test_full,
        )
        out["provider"] = provider
        final_rows.append(out)
    final_df = pd.DataFrame([
        {
            "provider": r["provider"],
            "family": r["family"],
            "name": r["name"],
            "n_models": r["n_models"],
            "models": json.dumps(r["models"], ensure_ascii=False),
            "selection_MAE": r["selection_MAE"],
            "MAE_typical": r["MAE_typical"],
            "RMSE_typical": r["RMSE_typical"],
            "MdAE_typical": r["MdAE_typical"],
            "MAE_full": r["MAE_full"],
            "RMSE_full": r["RMSE_full"],
            "MdAE_full": r["MdAE_full"],
            "weights": json.dumps(r.get("weights", []), ensure_ascii=False),
        }
        for r in final_rows
    ]).sort_values(["selection_MAE", "MAE_typical", "MAE_full"]).reset_index(drop=True)
    final_df.to_csv(RUN_DIR / f"{provider}_ofat_dl_ensemble_leaderboard.csv", index=False)
    final_df.to_csv(ART_DIR / f"{provider}_ofat_dl_ensemble_leaderboard_latest.csv", index=False)
    leaderboards.append(final_df)
    best_rows.append(final_df[final_df["n_models"] >= 2].head(1) if (final_df["n_models"] >= 2).any() else final_df.head(1))
    print(f"\nBest {provider} OFAT DL ensembles:")
    display(final_df.head(10))

all_leaderboard = pd.concat(leaderboards, ignore_index=True).sort_values(["provider", "selection_MAE", "MAE_typical"]).reset_index(drop=True)
best_provider = pd.concat(best_rows, ignore_index=True).sort_values(["provider"]).reset_index(drop=True)
save_df(all_leaderboard, "all_ofat_provider_dl_ensemble_leaderboard.csv")
save_df(best_provider, "best_ofat_provider_dl_ensembles.csv")
chart = best_provider[["provider", "selection_MAE", "MAE_typical", "MAE_full", "models", "weights"]].rename(columns={"selection_MAE": "ofat_dl_ensemble_selection_MAE"})
save_df(chart, "chart_ofat_dl_ensemble_provider_values.csv")

summary = {
    "run_name": RUN_NAME,
    "run_id": RUN_ID,
    "data_path": str(DATA_PATH),
    "artifacts_root": str(ART_DIR),
    "run_dir": str(RUN_DIR),
    "selected_dl_models": SELECTED_DL_MODELS,
    "providers": sorted(best_provider["provider"].unique().tolist()),
    "blend_train_frac": BLEND_TRAIN_FRAC,
    "blend_fit_frac": BLEND_FIT_FRAC,
    "max_subset_size": MAX_SUBSET_SIZE,
    "top_refit_specs": TOP_REFIT_SPECS,
    "best_by_provider": best_provider.to_dict(orient="records"),
}
(RUN_DIR / "best_ofat_dl_ensemble_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
(ART_DIR / "best_ofat_dl_ensemble_summary_latest.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved to:", RUN_DIR)
display(best_provider)


## Quick report

In [ ]:
if 'best_provider' in globals() and len(best_provider):
    plt.figure(figsize=(10, 4), dpi=130)
    plot_df = best_provider.copy()
    sns.barplot(data=plot_df, x="MAE_typical", y="provider", hue="family", dodge=False)
    plt.title("Best OFAT DL ensemble by provider")
    plt.xlabel("Typical holdout MAE")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

    display(best_provider[["provider", "family", "name", "n_models", "selection_MAE", "MAE_typical", "MAE_full", "models", "weights"]])
